# DADA2: 04 — Evaluation

Three TODOs.

You will conclude the assignment by running a RAG eval.

Three questions to answer with numbers:
- Is the decision model as good a router as the chat model? 
- Does the diagnosis agent beat the ladder, and at what cost?
- which parts of what you built were worth building?

In [1]:
from dotenv import load_dotenv
load_dotenv()

import time
from rag import config, decide, evaluate, retrieval, agent, ladder

## Read this first

This notebook is where the Gemini quota goes. The ladder agent spends about
five chat calls per question and yours about three, and the judge spends one
more on each answer. Fourteen questions through both agents is roughly 150
calls against a limit of 500 a day.

So, you will want to run it once, when notebooks 02 and 03 already pass their final cells. The comparison cells sleep between questions on purpose. Start them and leave them alone.

In [2]:
specific = evaluate.load_evalset("specific")
comparative = evaluate.load_evalset("comparative")
items = specific + comparative

print(f"{len(specific)} specific + {len(comparative)} comparative = {len(items)} labeled questions")
print("reranker:       cohere")
print("chat model:    ", config.CHAT_MODEL)
print("decision model:", config.DECISION_MODEL)

14 specific + 10 comparative = 24 labeled questions
reranker:       cohere
chat model:     gemini-3.5-flash-lite
decision model: clef-flash


In [9]:
import inspect
from rag import agent as A
dim = len(config.embeddings().embed_query("probe"))
print(f"embeddings:   {config.EMBEDDINGS} {config.EMBED_MODEL}, {dim} dims, index {config.index_name(dim)}")
print(f"chunks:       {config.CHUNK_SIZE} chars, {config.CHUNK_OVERLAP} overlap")
print(f"retrieval:    K={config.K}, POOL={config.POOL}, RRF_K={config.RRF_K}, "
      f"rewrites={inspect.signature(retrieval.write_queries).parameters['n'].default}")
print(f"agent:        SURE={A.SURE}, ABSENT_SURE={A.ABSENT_SURE}, MAX_ATTEMPTS={A.MAX_ATTEMPTS}")

embeddings:   cloudflare @cf/qwen/qwen3-embedding-0.6b, 1024 dims, index dada2-1024
chunks:       1200 chars, 150 overlap
retrieval:    K=5, POOL=10, RRF_K=60, rewrites=4
agent:        SURE=0.6, ABSENT_SURE=0.8, MAX_ATTEMPTS=3


## 1. Two routers

### TODO 4.1 — the decision model against the chat model (1 pt)

Every labeled question has a kind: the 14 in `evalset.json` are specific,
the 10 in `evalset_hard.json` are comparative. Ask both models which kind each
one is, and keep the time.

Write the chat-model version yourself. It is a `with_structured_output` call
with a `Literal` field, the same thing you did for `Filter` in notebook 02.
Use `decide.SCOPES` for the descriptions so both models are asked the same
question.

**Steps:**

1. Define a scoper that uses Scope to get structured output from chat model
3. In the loop below, the decision-model line is written. Add the same two lines for the chat model, keyed `"chat"`: time the call, add the seconds, add one to `right` when the choice matches `kind`.


In [3]:
from typing import Literal
from pydantic import BaseModel, Field
from rag.util import retry

# TODO 4.1 — a scoper that uses Scope to get structured output from chat model
class Scope(BaseModel):
    """Which kind of question this is."""
    scope: Literal["specific", "comparative", "figure"] = Field(
        description="; ".join(f"{k}: {v}" for k, v in decide.SCOPES.items()))

def chat_scope(question):
    t = time.time()
    out = retry(scoper.invoke, "Question about SEC 10-K filings: " + question
                + "\nWhat does it ask for?")
    return out.scope, time.time() - t

scoper = config.chat().with_structured_output(Scope)

In [4]:
labeled = [(i["question"], "specific") for i in specific] + \
           [(i["question"], "comparative") for i in comparative]

right = {"decision": 0, "chat": 0}
secs = {"decision": 0.0, "chat": 0.0}
for q, kind in labeled:
    t = time.time(); choice, _ = decide.scope(q); secs["decision"] += time.time() - t
    right["decision"] += (choice == kind)
    t = time.time(); choice, _ = chat_scope(q); secs["chat"] += time.time() - t
    right["chat"] += (choice == kind)
    time.sleep(3)

n = len(labeled)
for name in ("decision", "chat"):
    print(f"{name:9} {right[name]}/{n} right, {secs[name]/n:.2f}s per call")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


decision  20/24 right, 0.43s per call
chat      19/24 right, 0.89s per call


NOTE: Accuracy is close, and the chat model is probably not
slower. If you expected the decision model to win on both, write down what it does give you instead: a probability per option, and a call that does not count against the daily limit. Whether those are worth having is the question
the next section answers.

## 2. Two agents

### TODO 4.2 — the ladder against the diagnosis (1 pt)

`evaluate.compare_agents` runs each agent over the questions, has the chat
model judge every answer, and keeps the bill: chat calls, decision calls, and
seconds per question. Both agents expose `run(question)` returning the final
state, which is all it needs.

Fourteen questions is enough. Use eight specific and six comparative so both
kinds are in the mix.

**Steps:** nothing to write. Run the cell once, when both agents pass their final cells. If a 429 cuts it off, score the two agents in separate cells rather than restarting the pair.

This will take a while. Plan accordingly. 

Make sure your rate limit is reset or use a different provider (e.g., Groq, OpenRouter) instead of the Gemini API. 

Let it run and come back to see your results.


In [5]:
# TODO 4.2
subset = specific[:8] + comparative[:6]

rows = evaluate.compare_agents({"ladder": ladder.run, "diagnosis": agent.run}, subset)

  agent         accuracy  chat/q  decision/q    s/q
  --------------------------------------------------


/Users/mandar/agentic-ai-course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  ladder            0.93     4.9         0.0   18.5
  diagnosis         0.64     2.9         2.4   11.0


In [6]:
# the judge's reasons, on anything either agent got wrong
_, notes = evaluate.score_agent(agent.run, subset[:4])
for note in notes:
    mark = "ok " if note["correct"] else "WRONG"
    print(f"[{mark}] {note['question'][:66]}  ({note['chat_calls']} chat, {note['decision_calls']} decision)")
    print(f"        {note['reason'][:150]}\n")

[ok ] What does MU state that the preparation of financial statements an  (2 chat, 2 decision)
        CORRECT
The provided answer accurately identifies the management requirement regarding the preparation of financial statements under U.S. GAAP as stat

[ok ] What termination fee is AMD required to pay ZT Systems under certa  (2 chat, 2 decision)
        CORRECT
The answer correctly identifies the $300 million termination fee related to regulatory approvals for the ZT Systems acquisition as stated in t

[ok ] What factors does NVIDIA believe are the principal competitive fac  (2 chat, 2 decision)
        CORRECT
The provided answer accurately lists the principal competitive factors stated in NVIDIA's fiscal year 2025 filings.

[ok ] What competitive pressures from Intel and Nvidia did AMD face in f  (2 chat, 2 decision)
        CORRECT
The answer accurately and comprehensively identifies the competitive pressures AMD faced from Intel and Nvidia in fiscal year 2025 as supporte



In [7]:
for item in subset[4:]:
    q = item["question"]
    state = agent.run(q)
    ok, reason = evaluate.judge_answer(q, state["answer"])
    sources = sorted({s for s, _ in state.get("documents", [])})
    vp = state.get("probabilities", {})
    print(f"[{'ok ' if ok else 'WRONG'}] {q[:70]}")
    print(f"   route: {state['scope']} -> {state['strategy']}   "
          f"verdict: {state.get('verdict')} ({vp.get(state.get('verdict'), 0):.2f})   "
          f"attempts: {state['attempts']}")
    print(f"   gold:  {evaluate.gold_of(item)}")
    print(f"   found: {sources}")
    print(f"   answer: {state['answer'][:120]}")
    print(f"   judge:  {reason[:120]}\n")
    time.sleep(4)

[ok ] What forms the basis for AMD's estimates and judgments used in prepari
   route: specific -> filter   verdict: sufficient (0.58)   attempts: 1
   gold:  ['AMD_2025_mdna.txt']
   found: ['AMD_2025_mdna.txt', 'AMD_2025_risk_factors.txt']
   answer: AMD bases its estimates on historical experience and on various other assumptions that it believes to be reasonable unde
   judge:  CORRECT
The answer accurately identifies that AMD bases its estimates on historical experience and reasonable assumption

[WRONG] What financial charge did NVIDIA incur in fiscal year 2026 due to expo
   route: specific -> filter   verdict: sufficient (0.90)   attempts: 1
   gold:  ['NVDA_2026_mdna.txt']
   found: ['NVDA_2026_business.txt', 'NVDA_2026_mdna.txt', 'NVDA_2026_risk_factors.txt']
   answer: In fiscal year 2026, NVIDIA incurred a $4.5 billion charge associated with the H20 product for excess inventory and purc
   judge:  INCORRECT
Reason: The provided answer refers to fiscal year 2026 and future/h

In [8]:
print(ladder.ask("What target markets does NVIDIA include in its business for fiscal year 2024?"))

Based on the provided passages, the text mentions that NVIDIA's "target markets remain competitive" [NVDA_2024_risk_factors.txt], but the passages do not specifically list or name the specific target markets that NVIDIA includes in its business for fiscal year 2024. Therefore, the passages do not support a detailed answer to the question.


If the diagnosis agent got something wrong that the ladder got right, find
out which half failed. Run `decide.scope` and `decide.diagnose` on that
question by hand and look at the probabilities. A wrong route and a wrong
verdict are different bugs, and the fix for one is a threshold while the fix
for the other is a search strategy.

### TODO 4.3 — write it up (2 pts)

In the cell below, in your own words. Four or five short paragraphs.

**Conditions.** Reranker, chat model, decision model, k, chunk size. Enough
for someone to re-run you.

**The routers.** Which was more accurate, which was faster, and by how much.
Then say what the decision model is for, given those numbers.

**The agents.** Accuracy, chat calls per question, seconds per question, for
both. Name the question where they differed most and say why.

**One thing that did not help.** Something you built that the numbers say you
could remove. Say what the mechanism is, not that the model is better at it.

**What you would ship**, and what you would want to know about your users'
questions first.

Two things that cost marks: an improvement the table does not show, and a
difference explained by "the model is better" when the reason is in your
code.

**Steps:** write it as prose, four or five short paragraphs. The bold headings in the cell below are the order; replace the cell's text under each. Every number you quote should be in a table above.


---

### Your write-up

**Conditions.** Chat model `gemini-3.5-flash-lite` (it also served as the judge); decision model `clef-flash` on Cloudflare Workers AI. Embeddings: `qwen3-embedding-0.6b` via Cloudflare, 1024 dimensions, Pinecone index `dada2-1024`. Chunks of 1200 characters with 150 overlap. K = 5 passages to the model; the reranker is Cohere `rerank-v3.5`, reordering a pool of 10 down to 5. Multi-query uses 4 rewrites fused with RRF (constant 60). Agent thresholds were left at their defaults: SURE 0.6, ABSENT_SURE 0.8, MAX_ATTEMPTS 3. The routers were compared on all 24 labeled questions (14 specific, 10 comparative); the agents on 14 (`specific[:8] + comparative[:6]`), each run once.

**The routers.** The decision model labeled 20 of 24 questions correctly and the chat model 19 of 24, so accuracy differs by a single question. The decision model was about twice as fast (0.43 s against 0.89 s per call). The bigger difference is cost: its calls do not count against the 500-a-day Gemini quota, and it returns a probability for every option, which is what lets the graph fall back to a default strategy when it is unsure. In my traced run, the Micron memory-pricing question, which is labeled comparative, was routed as specific and still found both gold files, because the extractor named the company and left the year empty. Classifying questions is what the decision model is for: the same accuracy, faster, and free of the chat quota.

**The agents.** The ladder scored 0.93 with 4.9 chat calls and 18.5 s per question; the diagnosis agent scored 0.64 with 2.9 chat calls, 2.4 decision calls and 11.0 s. The gap is four questions out of fourteen. In my traced run, none of its failures came from routing. The clearest failure was "What target markets does NVIDIA include in its business for fiscal year 2024?": a passage from the gold file `NVDA_2024_business.txt` was retrieved, but on the third attempt the decision model returned `absent` at only 0.50. My `diagnose` overrides an unsure `absent` only while attempts remain, so at the cap it was accepted, and `answer` returned a fixed "the filings do not cover that" without a chat call. Run separately, the ladder did not name the markets either; it found only a passage saying its target markets remain competitive, cited it, and said the passages did not list them. So on this question a gold-file hit did not mean the answering passage was retrieved, and the two agents differed in how they failed: the ladder showed what it had found, while mine discarded it. A second failure, on NVIDIA's export controls over three years, went the other way: the verdict was `sufficient` at 0.45, nothing checks the confidence of a `sufficient` verdict, and the agent answered after one search. A third was the judge's mistake rather than the agent's: the $4.5 billion H20 answer matches the reference, but the judge sees neither the passages nor the reference, and it rejected fiscal 2026 as hypothetical. In the inspection cell the agent answered 4 of 4 correctly, and in my traced run of the other ten it answered 7 of 10, so 11 of 14 across those two runs against 9 of 14 in the table: the score moved by two questions between runs.

**What did not help.** The `absent` shortcut in `answer`. It saves one chat call when it fires, and in my traced run it fired only on a question the corpus answers, even though a passage from the gold file was already in `documents`. The ABSENT_SURE guard does not protect the last attempt, which is exactly where an unsure `absent` is most likely to appear after two failed searches. Removing the shortcut costs one chat call on those questions, and the answer prompt already tells the model to say in one sentence when the passages do not cover the question. The ladder's run on the same question shows what the alternative looks like: a cited answer that says what the passages do and do not contain.

**What I would ship.** On these numbers, the ladder: it was right on four more questions, and its extra cost (2 more chat calls and 7.5 s per question) is affordable at this scale. The diagnosis agent's speed and quota savings are real, but I would only ship it after removing the `absent` shortcut and adding a minimum confidence for `sufficient`, and then measuring it again. Those fixes are not shown in any table here. Before choosing, I would want to know what share of users' questions are comparative, since both agent failures I traced came from a verdict on retrieved passages; how often they ask about things the filings truly do not cover, which is the only case the `absent` exit is good for; and I would give the judge the reference answers, since it misgraded a correct answer in this run.

**Copy your write-up to README.md.**